### 텐서플로 2.0 

pip install tensorflow
간단한 구조의 인공 신경망으로 1차 선형회귀식을 찾는 문제를 살펴보자.
텐서플로 라이브러리를 tf란 이름으로 불러온다. __version__ 속성으로 버전을 확인한다.

In [2]:
import tensorflow as tf
print(tf.__version__)

2.21.0


### x 변수, y 변수 데이터 

In [3]:
# 모델 학습에 사용할 입력 데이터를 준비한다. y=x+1 관계를 갖는 숫자를 x,y 변수에 각각 10개씩 입력한다.
# x변수의 숫자 배열을 (10행, 1열)형태의 2차원 배열로 변환한다. 

import pandas as pd
import numpy as np

x = [-3,  31,  -11,  4,  0,  22, -2, -5, -25, -14]
y = [ -2,   32,   -10,   5,  1,   23,  -1,  -4, -24,  -13]

X_train = np.array(x).reshape(-1, 1)
y_train = np.array(y)

print(X_train.shape, y_train.shape)

(10, 1) (10,)


### 인공 신경망 모델 

케라스 Sequential API는 레이어 여러 개를 연결하여 신경망 모델을 구성하는 도구이다.
간단한 아키텍처를 가지면서도 대부분의 딥러닝 모델을 만들 수 있다는 장점이 있다.

1개의 레이어만 사용하여 가장 단순한 형태의 신경망을 만들어 보자.
Sequential 모델 인스턴스를 생성하고 add 메소드를 사용하여 완전 연결 레이어(Dense)를 모델에 추가한다.
입력 데이터의 차원(input_dim)은 모델 학습에 사용하는 설명 변수(피처)의 개수를 지정한다.
여기서는 1개의 피처를 사용하므로 1로 설정한다.

In [ ]:
# 완전 연결 레이어의 출력값은 목표레이블(Y)를 예측한다.
# 한 개의 연속형 수치(주택 가격)를 예측하는 회귀 문제이므로 유닛(unit) 개수는 1이다.
# 활성화 함수로 'linear'옵션을 지정하여 선형 함수의 출력을 그대로 사용한다.

from tensorflow.keras import Sequential
from tensorflow.keras.layers import Dense

model = Sequential()        # 입력부터 출력까지 순차적으로 연결되는 구조를 만든다. : Sequential()
model.add(Dense(units=1, activation='linear', input_dim=1)) # 하나의 층을 모델에 추가하는 명령
# units=1 : 출력되는 숫자의 개수가 1개다.

In [7]:
# summary 메소드를 이용하여 모델 아키텍처(구조)를 확인한다.
# 딥러닝 모델이 학습할 모수(파라미터:param #)는 2개이다. 일차함수의 기울기(회귀계수)와 절편(상수항)이다.

model.summary()

Model: "sequential_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ dense_1 (Dense)                 │ (None, 1)              │             2 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 2 (8.00 B)

 Trainable params: 2 (8.00 B)

 Non-trainable params: 0 (0.00 B)

In [8]:
# 모델이 훈련하는데 필요한 기본 설정을 compile 함수에 지정한다.
# 옵티마이저와 손실 함수를 설정한다. adam 옵티마이저를 선택하고 회귀 분석의 손실 함수인 평균제곱오차를 지정한다.
# metrics 옵션에 보조 평가 지표를 추가할 수 있다. 여기서는 평균 절대오차(mae) 를 추가하여 손실 함수를 모니터링할 때 함께 추적하기로 한다.

model.compile(optimizer='adam', loss='mse', metrics=['mae'])

## 모델 학습 및 예측

fit 메소드에 훈련 데이터를 입력하여 모델을 학습시킨다. 컴파일 단계에서 설정한 adam 옵티마이저와 mse 손실 함수를 가지고 최적의 가중치와 편향을 찾는다.
에포크(epoch)는 전체 입력 데이터를 모두 몇 번 학습할 것인지 반복 횟수를 정한다.
verbose 옵션을 False(0)로 지정하면 훈련 과정을 화면에 보여주지 않는다. 훈련 과정을 표시하려면 1 또는 2를 입력한다.

In [ ]:
model.fit(X_train, y_train, epochs=3000, verbose=0)

In [10]:
# 학습을 마친 딥러닝 모델의 가중치를 확인하려면 weights 속성을 보면 된다.
# 기울기에 해당하는 가중치(kernel:0)와 절편에 해당하는 편향(bias:0) 모두 1에 가까운 값을 갖는다.
# 따라서 모델 학습을 통해 일차함수 관계식을 매우 근사하게 찾아낸 것으로 볼 수 있다.

model.weights

[<Variable path=sequential_1/dense_1/kernel, shape=(1, 1), dtype=float32, value=[[0.99999666]]>,
 <Variable path=sequential_1/dense_1/bias, shape=(1,), dtype=float32, value=[0.99940336]>]

### 예측

In [ ]:
# 테스트 데이터(X)를 predict 메소드에 입력하면 목표 레이블(Y)에 대한 예측값을 얻을 수 있다.
import numpy as np
x_list = [[11],[12],[13]]
x_new = np.array(x_list)

model.predict(x_new)

ValueError: Unrecognized data type: x=[[11], [12], [13]] (of type <class 'list'>)